### 工具封装

In [1]:
from langchain.tools import tool

In [4]:
@tool
def multiply(a: int, b: int) -> int:
    """乘法计算器"""
    return a * b
print(multiply.name)
print(multiply.description)
print(multiply.args)
print(multiply.args_schema.model_json_schema())

multiply
乘法计算器
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
{'description': '乘法计算器', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


In [ ]:
#天气查询工具封装
#pip install requsets==2.31.4
import requests

def get_weather(city: str) -> str:
    """查询天气"""
    url = "https://eolink.o.apispace.com/456456/weather/v001/now"
    payload = {"areacode" : "101030600"}
    headers = {
        "X-APISpace-Token":"l41bgca4135bkbyulikieqyu8uww8qdr"
    }
    response=requests.get(url,params=payload, headers=headers)
    return print(response.text)
get_weather("天津北辰")


{"status":0,"result":{"location":{"areacode":"101030600","name":"北辰区","country":"中国","path":"北辰区,天津市,天津市,中国"},"realtime":{"text":"雾","code":"18","temp":17.4,"feels_like":18,"rh":84,"wind_class":"1级","wind_speed":0.8,"wind_dir":"东南风","wind_angle":147,"prec":0.0,"prec_time":"2026-04-17 20:00:00","clouds":0,"vis":3000,"pressure":1010,"dew":14,"uv":0,"weight":2,"brief":"雾气缭绕","detail":"雾气缭绕，出行注意安全哦"},"last_update":"2026-04-17 21:03"}}


In [ ]:
# 获取城市编码
# pip install pandas==2.3.1
import pandas as pd
city_df = pd.read_csv("city.csv")
def get_city_code(city_name:str) ->int:
    '''
    获取城市编码
    '''
    # 优先匹配区县
    match = city_df[city_df['district']==city_name]
    if not match.empty:
        return match.iloc[0]['areacode/城市ID']
    # 匹配城市
    match = city_df[city_df['city']==city_name]
    if not match.empty:
        return match.iloc[0]['areacode/城市ID']
    # 匹配省份
    match = city_df[city_df['city'].str.contains(city_name,na=False)]
    if not match.empty:
        return match.iloc[0]['areacode/城市ID']
    #默认北京
    return 101010100
city_df.head()

In [ ]:
#天气查询工具封装
#pip install requsets==2.31.4
import requests
from langchain.tools import tool
@tool
def get_weather(city: str) -> str:
    """查询天气"""
    url = "https://eolink.o.apispace.com/456456/weather/v001/now"
    city_code = get_city_code(city)
    payload = {"areacode" : city_code}
    headers = {
        "X-APISpace-Token":"l41bgca4135bkbyulikieqyu8uww8qdr"
    }
    response=requests.get(url,params=payload, headers=headers)
    #将结果转换为json数据
    data = response.json()
    temp = data.get("result").get(realtime).get("temp")
    wd = data.get("result").get(realtime).get("text")
    return f'天气：{wd}，温度：{temp}℃'
get_weather("天津北辰")


### 调用工具

In [8]:
#智普ai
import os
from langchain_community.chat_models import ChatZhipuAI
key = 'b96d974eb5eb4648b9cdc76fbf423ddd.1bg7tnqu3cyIlpVg'
os.environ["ZHIPUAI_API_KEY"] = key
chat = ChatZhipuAI(
    model="glm-4",
    temperature=0.5, #模型温度  值在0-1之间 值越小 随机性越低
)

In [ ]:
from langchain.agents import create_react_agent,AgentExecutor
from langchain import hub
#创建工具对象
tools = [get_weather,multiply]
#获取提示词
prompt = hub.pull("huchase17/react")
#创建智能体
agent = create_react_agent(llm=chat,tools=tools,prompt=prompt)
#创建AgentExecutor 运行智能体
agent_executor = AgentExecutor(agent=agent,tools=tools,verbose=True) #verbose=True 打印调试过程
#调用智能体
agent_executor.invoke({'input ': "请帮我查询一下天津北辰的天气,顺便计算一下5乘以6等于多少？"})